# NIfTI scans to training slices

Builds the real-scan dataset used by `train.py` and zips it as `/kaggle/working/data_real.zip`:

```
data_real/Training/<class>/<patientID>_z###.jpg
data_real/Testing/<class>/<patientID>_z###.jpg
```

Each patient goes entirely into Training or Testing (20% of patients are held out), so test accuracy is not inflated by near-identical slices of the same patient. The conversion code lives in `nifti_slices.py` in the repo.

**Before running:** attach `vietanh21/brats-2021-task-1-dataset` and `dukhailly/meningioma` with *Add Input*, and turn **Internet on** in the notebook settings.

**The first code cell does everything** (find datasets, get the code, convert, zip). The cell after it is an optional preview. Each cell works on its own, so a kernel restart or running one cell alone is fine.

In [ ]:
# ---------- Build data_real.zip (this one cell does the whole job) ----------
import glob, os, shutil, subprocess, sys
from IPython.display import FileLink, display

OUT_DIR = "/kaggle/working/data_real"
ZIP_BASE = "/kaggle/working/data_real"          # .zip is added automatically
REPO_DIR = "/kaggle/working/repo"
SCRIPT = f"{REPO_DIR}/nifti_slices.py"
BRANCH = "main"
SLICES_PER_PATIENT = 8
MAX_PATIENTS = 400
EXTRA_ARGS = []   # e.g. ["--notumor_from_tumor_patients"] or ["--pituitary_root", "/kaggle/input/..."]


def find_input(slug):
    """Find an attached dataset folder by its slug, wherever Kaggle mounted it."""
    hits = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
            if os.path.basename(d.rstrip("/")) == slug and d.count("/") <= 7]
    if not hits:
        raise FileNotFoundError(
            f"Dataset '{slug}' is not attached. Add it with 'Add Input' on the right, then re-run.")
    return sorted(hits, key=len)[0].rstrip("/")


def run(cmd):
    """Run a shell command, show its output live, and stop if it fails."""
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


# 1. Datasets
GLIOMA_ROOT = find_input("brats-2021-task-1-dataset")
MENINGIOMA_ROOT = find_input("meningioma")
print("glioma:    ", GLIOMA_ROOT)
print("meningioma:", MENINGIOMA_ROOT)

# 2. Code
run(["rm", "-rf", REPO_DIR])
run(["git", "clone", "--depth", "1", "-b", BRANCH,
     "https://github.com/imfonggg/AI-for-Brain-Tumor-Detection.git", REPO_DIR])
run([sys.executable, "-m", "pip", "install", "-q", "nibabel"])

# 3. Convert (takes a while: IXI healthy scans are streamed over the internet)
run(["rm", "-rf", OUT_DIR])
run([sys.executable, "-u", SCRIPT, "--out_dir", OUT_DIR,
     "--glioma_root", GLIOMA_ROOT,
     "--meningioma_root", MENINGIOMA_ROOT,
     "--ixi",
     "--slices_per_patient", str(SLICES_PER_PATIENT),
     "--max_patients", str(MAX_PATIENTS),
     *EXTRA_ARGS])
n_images = len(glob.glob(f"{OUT_DIR}/*/*/*.jpg"))
if n_images == 0:
    raise RuntimeError("No slices were written. Check the log above for the class that found 0 cases.")
print(f"\nOK: {n_images} slices in {OUT_DIR}")

# 4. Zip
zip_path = shutil.make_archive(ZIP_BASE, "zip", OUT_DIR)
print(f"Saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
print("Download it from the Output panel on the right (/kaggle/working), or click the link:")
os.chdir("/kaggle/working")
display(FileLink(os.path.basename(zip_path)))

## Optional: add a brain metastasis class

Run this **after** the build cell above. It downloads the annotated brain metastasis dataset straight from Figshare ([Scientific Data paper](https://www.nature.com/articles/s41597-023-02123-0), CC0, 75 patients from 5 hospitals, contrast-enhanced T1 with tumor masks), adds `metastasis` slices to `data_real` using the same patient-level split, and re-zips `data_real.zip`.

- Needs Internet on. The image zip is about 8.5 GB, so the download takes a while; it goes to `/tmp` and is deleted after extracting.
- Check the printed patient IDs: each patient must have one ID across all their scans, or the patient split won't hold. Adjust `patient_id()` if the IDs look wrong.
- Adding a class changes the model's outputs, so **both models need a full retrain** on the new zip.
- Like the other sources, this comes from its own hospitals and scanners, so watch for the model learning the dataset instead of the tumor (see `docs/datasets.md`).

In [ ]:
# ---------- Optional: add metastasis slices (run after the build cell) ----------
import glob, os, re, shutil, subprocess, sys, zipfile
from collections import defaultdict
import requests
from IPython.display import FileLink, display

OUT_DIR = "/kaggle/working/data_real"
ZIP_BASE = "/kaggle/working/data_real"
REPO_DIR = "/kaggle/working/repo"
WORK_DIR = "/tmp/metastasis"                 # big download lives outside /kaggle/working
CLASS_NAME = "metastasis"
SLICES_PER_PATIENT = 8
STUDIES_PER_PATIENT = 1                      # scans per patient to use (patients have several follow-ups)
IMAGES_URL = "https://ndownloader.figshare.com/files/39604831"   # NIFTIs_Images.zip (~8.5 GB)
MASKS_URL = "https://ndownloader.figshare.com/files/39603652"    # NIFTIs_Masks.zip (~12 MB)

if not glob.glob(f"{OUT_DIR}/*/*/*.jpg"):
    raise RuntimeError("Run the build cell above first, so data_real exists.")

# Reuse the split and slicing helpers from the repo the build cell cloned
if not os.path.exists(f"{REPO_DIR}/nifti_slices.py"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/imfonggg/AI-for-Brain-Tumor-Detection.git", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=True)
sys.path.insert(0, REPO_DIR)
import numpy as np
import nifti_slices as ns


def download(url, path):
    if os.path.exists(path):
        print(f"already downloaded: {path}")
        return
    print(f"downloading {url} -> {path}", flush=True)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done, next_report = 0, 0
        with open(path + ".part", "wb") as f:
            for chunk in r.iter_content(chunk_size=8 << 20):
                f.write(chunk)
                done += len(chunk)
                if done >= next_report:
                    print(f"  {done / 1e9:.1f} / {total / 1e9:.1f} GB", flush=True)
                    next_report += 500 << 20
    os.rename(path + ".part", path)


def scan_key(name):
    """'.../X_img.nii.gz' and '.../X_msk.nii.gz' -> 'X' so images and masks pair up."""
    base = os.path.basename(name)
    return re.sub(r"_(img|msk)\.nii(\.gz)?$", "", base)


def patient_id(rel_path):
    """Patient ID for one scan. Uses the top-level folder if scans sit in per-patient
    folders, otherwise the part of the filename before the first underscore."""
    parts = rel_path.replace("\\", "/").split("/")
    pid = parts[0] if len(parts) > 1 else parts[0].split("_")[0]
    return re.sub(r"[^A-Za-z0-9-]", "-", pid)


os.makedirs(WORK_DIR, exist_ok=True)
masks_zip, images_zip = f"{WORK_DIR}/masks.zip", f"{WORK_DIR}/images.zip"
download(MASKS_URL, masks_zip)
with zipfile.ZipFile(masks_zip) as z:
    z.extractall(f"{WORK_DIR}/masks")
masks = {scan_key(p): p for p in glob.glob(f"{WORK_DIR}/masks/**/*_msk.nii*", recursive=True)}
print(f"{len(masks)} masks")

# Extract only images that have a mask, then free the disk space
download(IMAGES_URL, images_zip)
with zipfile.ZipFile(images_zip) as z:
    members = [m for m in z.namelist() if re.search(r"_img\.nii(\.gz)?$", m) and scan_key(m) in masks]
    print(f"extracting {len(members)} matching images")
    for m in members:
        z.extract(m, f"{WORK_DIR}/images")
os.remove(images_zip)

# Group scans by patient; keep the first STUDIES_PER_PATIENT scans of each
images = sorted(glob.glob(f"{WORK_DIR}/images/**/*_img.nii*", recursive=True))
if not images:
    raise RuntimeError("No *_img.nii files matched a mask. Look at the zip contents in /tmp/metastasis.")
root = os.path.commonpath([os.path.dirname(i) for i in images])   # skips wrapper folders in the zip
by_patient = defaultdict(list)
for img in images:
    by_patient[patient_id(os.path.relpath(img, root))].append(img)
print(f"{len(by_patient)} patients; first IDs: {sorted(by_patient)[:8]}")
print("scans per patient (first 8):", [len(by_patient[p]) for p in sorted(by_patient)[:8]])


class SplitArgs:                       # same split settings nifti_slices.py uses
    test_fraction, seed = ns.TEST_FRACTION, ns.SPLIT_SEED


shutil.rmtree(f"{OUT_DIR}/Training/{CLASS_NAME}", ignore_errors=True)
shutil.rmtree(f"{OUT_DIR}/Testing/{CLASS_NAME}", ignore_errors=True)
done = 0
for pid, scans in sorted(by_patient.items()):
    saved = 0
    for k, img in enumerate(scans[:STUDIES_PER_PATIENT]):
        try:
            vol, seg = ns.load_ras(img), ns.load_ras(masks[scan_key(img)])
        except Exception as e:
            print(f"  skipping {os.path.basename(img)}: {e}")
            continue
        area = (seg > 0).sum(axis=(0, 1))        # labels 1 and 2 are both tumor here
        if area.max() == 0:
            continue
        zs = ns.evenly_spaced(np.where(area >= 0.5 * area.max())[0], SLICES_PER_PATIENT)
        out = ns.class_dir(OUT_DIR, pid, CLASS_NAME, SplitArgs)
        # Keep '<pid>_z<digits>' so data_loader.py groups every scan of a patient together
        saved += sum(ns.save_axial(vol, z, f"{out}/{pid}_z{k}{z:03d}.jpg", 500) for z in zs)
    if saved:
        done += 1
        if done % 25 == 0:
            print(f"  {CLASS_NAME}: {done}/{len(by_patient)} patients", flush=True)
print(f"{CLASS_NAME}: saved slices from {done} patients")
ns.summarize(OUT_DIR)

zip_path = shutil.make_archive(ZIP_BASE, "zip", OUT_DIR)
print(f"Saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
os.chdir("/kaggle/working")
display(FileLink(os.path.basename(zip_path)))

In [ ]:
# ---------- Optional: preview 6 random slices per class ----------
import glob, os, random
import matplotlib.pyplot as plt
from PIL import Image

OUT_DIR = "/kaggle/working/data_real"
if not glob.glob(f"{OUT_DIR}/Training/*/*.jpg"):
    raise RuntimeError(f"No slices in {OUT_DIR} yet: run the cell above first.")

classes = sorted(os.listdir(f"{OUT_DIR}/Training"))
fig, axes = plt.subplots(len(classes), 6, figsize=(15, 2.7 * len(classes)), squeeze=False)
for row, cls in enumerate(classes):
    files = glob.glob(f"{OUT_DIR}/Training/{cls}/*.jpg")
    for col, f in enumerate(random.sample(files, min(6, len(files)))):
        axes[row, col].imshow(Image.open(f), cmap="gray")
        axes[row, col].set_title(f"{cls}\n{os.path.basename(f)[:22]}", fontsize=7)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()